# 🛒 Market Basket Analysis (Association Rules via Apriori)
Phân tích tập phổ biến và luật kết hợp để khám phá hành vi mua sắm đi kèm (Cross-sell/Up-sell), 
từ đó thiết kế các gói sản phẩm (Product Bundling) tối ưu doanh thu.

In [ ]:
import os
import warnings
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from mlxtend.frequent_patterns import apriori, association_rules
from mlxtend.preprocessing import TransactionEncoder

# Cấu hình hiển thị và tắt cảnh báo
pd.set_option('display.max_colwidth', None)
pd.set_option('display.max_columns', None)
warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")
plt.rcParams["font.family"] = "DejaVu Sans"

DATA_DIR = "data"

## 1. Nạp Dữ Liệu
Sử dụng relative path để đảm bảo tính di động của dự án.

In [ ]:
df_ecomsales = pd.read_csv(os.path.join(DATA_DIR, "EcomSales.csv"))
df_products = pd.read_csv(os.path.join(DATA_DIR, "Product.csv"))

print(f"Tổng số giao dịch gốc: {df_ecomsales['OrderID'].nunique():,}")

# Merge dữ liệu để lấy tên Subcategory
df = pd.merge(
    df_ecomsales[['OrderID', 'ProductCode', 'Quantity']], 
    df_products[['ProductCode', 'Subcategory']], 
    how='inner', 
    on='ProductCode'
)

## 2. Khám Phá Dữ Liệu (EDA - Exploratory Data Analysis)
*Tech Debt Fix:* Trực quan hóa dữ liệu cơ bản trước khi chạy thuật toán học máy để hiểu phân bổ giỏ hàng.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6), dpi=100)

# Biểu đồ 1: Top 10 Subcategory bán chạy nhất (Theo tần suất xuất hiện trong đơn)
top_items = df['Subcategory'].value_counts().head(10)
sns.barplot(x=top_items.values, y=top_items.index, palette="viridis", ax=axes[0])
axes[0].set_title("Top 10 Most Frequent Subcategories", fontweight="bold")
axes[0].set_xlabel("Frequency (Times bought)")

# Biểu đồ 2: Phân bổ kích thước giỏ hàng (Basket Size)
basket_sizes = df.groupby('OrderID')['Subcategory'].nunique()
sns.histplot(basket_sizes, bins=range(1, 15), discrete=True, color="coral", ax=axes[1])
axes[1].set_title("Distribution of Basket Sizes (Unique Items per Order)", fontweight="bold")
axes[1].set_xlabel("Number of Unique Items")
axes[1].set_ylabel("Number of Orders")
axes[1].set_xticks(range(1, 15))

plt.tight_layout()
plt.show()

## 3. Tiền Xử Lý & Chuẩn Bị Giỏ Hàng (Market Basket)
*Tech Debt Fix:* Sử dụng `.unique()` để khử trùng lặp (deduplicate) các sản phẩm cùng loại 
trong cùng một OrderID trước khi đưa vào TransactionEncoder, giúp giảm nhiễu thuật toán.

In [ ]:
# Khử trùng lặp và tạo list transactions
transactions = df.groupby('OrderID')['Subcategory'].unique().tolist()

# Encode dữ liệu sang ma trận Boolean (One-Hot Encoding)
te = TransactionEncoder()
te_ary = te.fit(transactions).transform(transactions)
df_encoded = pd.DataFrame(te_ary, columns=te.columns_)

print(f"Kích thước ma trận giỏ hàng (Orders x Items): {df_encoded.shape}")

## 4. Thuật Toán Apriori & Trích Xuất Luật Kết Hợp
*Tech Debt Fix:* Nâng `min_support` lên mức an toàn (`0.005` hoặc `0.01`) thay vì `0.001` 
để tránh rủi ro tràn RAM (Out-Of-Memory Crash) sinh ra hàng triệu frequent itemsets không có ý nghĩa.

In [ ]:
# 1. Tìm tập phổ biến (Frequent Itemsets)
# Sử dụng min_support = 0.005 (Sản phẩm phải xuất hiện trong ít nhất 0.5% tổng số đơn hàng)
frequent_itemsets = apriori(df_encoded, min_support=0.005, use_colnames=True)
print(f"Tổng số Frequent Itemsets tìm được: {frequent_itemsets.shape[0]}")

# 2. Sinh luật kết hợp (Association Rules)
rules = association_rules(frequent_itemsets, metric="confidence", min_threshold=0.1)
rules = rules[rules['lift'] > 1.2] # Chỉ lấy các luật có Lift > 1.2 (Sức mạnh liên kết cao)

# 3. Làm sạch output để dễ đọc
rules['Combo'] = rules['antecedents'].apply(lambda x: ', '.join(list(x))) + "  --->  " + rules['consequents'].apply(lambda x: ', '.join(list(x)))
top_20_rules = rules.sort_values('lift', ascending=False).head(20)

top_20_rules[['Combo', 'support', 'confidence', 'lift']].round(4).reset_index(drop=True)

## 5. Báo Cáo Phân Tích & Đề Xuất Kinh Doanh

---

### 🏆 1. Sức Mạnh Liên Kết Lớn Nhất (Top Combo)
* **Combo:** `[Nến thơm, Xịt phòng, Mặt nạ] ---> [Đồ làm móng, Tinh dầu/Bom tắm]`
* **Lift = 4.25**: Khách hàng mua nhóm A có tỷ lệ mua thêm nhóm B cao **gấp 4.25 lần** so với việc mua ngẫu nhiên.
* **Confidence**: Cứ 10 người mua nhóm (Nến + Mặt nạ + Xịt phòng) thì có đến **4.5 người** chốt mua thêm đồ làm móng hoặc bom tắm ngay trong cùng một đơn hàng.

---

### 🛍️ 2. Phát Hiện Hai "Hệ Sinh Thái" Mua Sắm Tiêu Biểu
* **Cụm 1 - "Home Spa" (Thư giãn tại gia):** Nến thơm + Mặt nạ + Tinh dầu tắm + Đồ làm móng.
* **Cụm 2 - "Beauty from Within" (Khỏe đẹp toàn diện):** Vitamin/Thực phẩm chức năng liên tục được mua kèm với Kem dưỡng tay, Dầu gội và Dưỡng da mặt.

---

### 🧠 3. Tâm Lý Khách Hàng (Customer Psychology Insight)
* **Tâm lý "Thư giãn cuối tuần":** Khách hàng không mua các sản phẩm rời rạc, họ đang chi trả cho một *"trải nghiệm"*. Việc giỏ hàng có nến và mặt nạ chứng tỏ họ có chủ đích set-up một buổi tối thư giãn. Việc thuật toán phát hiện Tinh dầu tắm đi kèm là mảnh ghép hoàn hảo cho trải nghiệm này.
* **Tâm lý "Chăm sóc tiểu tiết":** Tệp khách mua Vitamin là những người cực kỳ kỷ luật về sức khỏe. Việc họ mua kèm "kem dưỡng tay" (một vùng da hay bị bỏ quên) cho thấy đây là tệp khách hàng kỹ tính, sẵn sàng chi trả cao cho các giải pháp chăm sóc từ gốc rễ.

---

### 🎯 4. Đề Xuất Hành Động Kinh Doanh (Actionable Insights)
1. **Đóng gói Combo (Product Bundling):** Tạo hộp quà *"Weekend Reset Kit"* gồm: Nến thơm + Bom tắm + Mặt nạ + Bộ sơn móng. Chiết khấu 10% khi mua cả bộ nhằm tăng mạnh giá trị trung bình đơn hàng (AOV).
2. **Tối ưu Bán chéo (Cross-sell) trên Website/App:** Cấu hình thuật toán gợi ý: Khi khách thêm *'Nến thơm'* vào giỏ, pop-up tự động hiện: *"Thêm chút thư giãn với Bom tắm/Tinh dầu tắm"* thay vì gợi ý một loại nến khác.
3. **Định hướng Content Marketing:** Triển khai chuỗi video Short/Reels với chủ đề: *"Quy trình 4 bước tự thưởng bản thân sau 1 tuần chạy deadline"*, lồng ghép khéo léo 4 sản phẩm thuộc cụm Home Spa để định hướng nhu cầu.